In [2]:
import os
os.chdir('/data2/zhouwg_data/project/Garfield')
os.getcwd()

In [3]:
# load packages
import os
import warnings
import scanpy as sc
from mudata import MuData
warnings.simplefilter(action="ignore", category=FutureWarning)

In [4]:
rna = sc.read_h5ad('../Garfield_test/data/human_pbmc_10x_3k_RNA.h5ad')
rna.layers['counts'] = rna.X.copy()
rna.obs['batch'] = 'RNA'
atac = sc.read_h5ad('../Garfield_test/data/human_pbmc_10x_3k_ATAC.h5ad')
atac.layers['counts'] = atac.X.copy()
atac.obs['batch'] = 'ATAC'
mdata = MuData({"rna": rna, "atac": atac})
mdata

In [5]:
import warnings
import io
import pybedtools
import pkgutil
import numpy as np
import pandas as pd
import anndata as ad
from scipy.sparse import (
    issparse,
    csr_matrix,
    coo_matrix,
    diags,
)
from collections import defaultdict, Counter
from pandas.api.types import is_numeric_dtype
from sklearn.feature_extraction.text import TfidfTransformer
from scipy.sparse.linalg import svds
from sklearn.cross_decomposition import CCA
from sklearn.utils.extmath import randomized_svd


def get_centroids(sub_data, labels):
    """
    Compute the centroids (cluster mean) of adata.

    Parameters
    ----------
    sub_data: scanpy object of each sample
    labels: np.array of shape (n_samples,)
        Cluster labels of each sample, coded from 0, ..., num_clusters-1

    Returns
    -------
    centroids: np.array of shape (n_centroids, n_features)
        Matrix of cluster centroids, the i-th column is the centroid of the i-th cluster
    """
    arr = sub_data.X
    arr_raw = sub_data.layers["counts"]
    meta = sub_data.obs
    cluster_label_to_indices = defaultdict(list)
    for i, l in enumerate(labels):
        cluster_label_to_indices[l].append(i)

    unique_labels = sorted(cluster_label_to_indices.keys())
    if not all(i == l for i, l in enumerate(unique_labels)):
        raise ValueError('labels must be coded in integers from 0, ..., n_clusters-1.')

    ## 对于每个簇，计算其均值，返回标准化的矩阵
    centroids = np.empty((len(unique_labels), arr.shape[1]))
    for curr_label, indices in cluster_label_to_indices.items():
        centroids[curr_label, :] = arr[indices, :].mean(axis=0)

    ## 对于每个簇，计算其均值，返回raw counts的矩阵
    counts = np.empty((len(unique_labels), arr.shape[1]))
    for curr_label, indices in cluster_label_to_indices.items():
        counts[curr_label, :] = arr_raw[indices, :].mean(axis=0)

    meta_info = np.empty((len(unique_labels), meta.shape[1]),
                         dtype=object)  # dtype=object 的意思是在 NumPy 数组中使用 Python 对象作为数据类型
    for curr_label, indices in cluster_label_to_indices.items():
        for col_idx in range(meta.shape[1]):
            if is_numeric_dtype(meta.iloc[indices, col_idx].dtype):
                # 如果是数值型数据
                meta_info[curr_label, col_idx] = meta.iloc[indices, col_idx].mean(axis=0)
            else:
                # 如果是字符型或其他类型 则通过投票找到最可能的label
                true_labels = meta.iloc[:, col_idx]  # .tolist()
                res = summarize_clustering(labels, curr_label, true_labels)
                meta_info[curr_label, col_idx] = res

    meta_info = pd.DataFrame(meta_info, columns=sub_data.obs.columns)

    for i, column in enumerate(meta_info.columns):
        if type(meta_info.iloc[:, i][0]) == list:
            meta_info.iloc[:, i] = sum(meta_info.iloc[:, i], [])
        else:
            meta_info.iloc[:, i] = meta_info.iloc[:, i]

    # use scanpy functions to do the adata_sub construction
    adata_sub = ad.AnnData(centroids, obs=meta_info, dtype=np.float32)
    adata_raw = ad.AnnData(counts, dtype=np.float32)
    adata_sub.layers["counts"] = adata_raw.X.copy()
    adata_sub.var_names = sub_data.var_names
    del adata_raw
    return adata_sub

## summarize_clustering
def summarize_clustering(clustering, curr_label, true_labels):
    """
    Compute the majority cell type for each cluster.

    Parameters
    ----------
    clustering: np.array of shape (n_samples,)
        Clustering labels, coded from 0, 1, ..., n_clusters.
    true_labels: np.array of shape (n_samples,)
        Groundtruth labels.

    Returns
    -------
    np.array of shape (n_clusters,)
        The majority voting results.
    """
    res = []
    cluster_label_to_indices = defaultdict(list)
    for i, l in enumerate(clustering):
        cluster_label_to_indices[l].append(i)

    # 使用字典推导式提取curr_label的子字典
    #     if len(curr_label) > 1:
    #         sub_dict = {key: cluster_label_to_indices[key] for key in curr_label}
    #     else:
    sub_dict = {curr_label: cluster_label_to_indices[curr_label]}

    for key in sub_dict.keys():
        curr_indices = sub_dict[key]  # sub_dict.get(key)
        curr_true_labels = true_labels.iloc[curr_indices]
        # majority voting
        # randomly break the ties
        curr_true_labels = np.random.permutation(curr_true_labels)
        counter = Counter(curr_true_labels)
        most_common_element, _ = counter.most_common(1)[0]
        res.append(most_common_element)

    return res


def cdist_correlation(arr1, arr2):
    """Calculate pair-wise 1 - Pearson correlation between X and Y.

    Parameters
    ----------
    arr1: np.array of shape (n_samples1, n_features)
        First dataset.
    arr2: np.array of shape (n_samples2, n_features)
        Second dataset.

    Returns
    -------
    array-like of shape (n_samples1, n_samples2)
        The (i, j)-th entry is 1 - Pearson correlation between i-th row of arr1 and j-th row of arr2.
    """
    n, p = arr1.shape
    m, p2 = arr2.shape
    assert p2 == p

    arr1 = (arr1.T - np.mean(arr1, axis=1)).T
    arr2 = (arr2.T - np.mean(arr2, axis=1)).T

    arr1 = (arr1.T / np.sqrt(1e-6 + np.sum(arr1 ** 2, axis=1))).T
    arr2 = (arr2.T / np.sqrt(1e-6 + np.sum(arr2 ** 2, axis=1))).T

    return 1 - arr1 @ arr2.T

def drop_zero_variability_columns(arr_lst: list, tol=1e-8):
    """
    Drop columns for which its standard deviation is zero in any one of the arrays in arr_list.

    Parameters
    ----------
    arr_lst: list of np.array
        List of arrays
    tol: float, default=1e-8
        Any number less than tol is considered as zero

    Returns
    -------
    List of np.array where no column has zero standard deviation
    """
    # assert all([arr_lst[i].shape[1] == arr_lst[0].shape[1] for i in range(len(arr_lst))])
    bad_columns = set()
    for arr in arr_lst:
        curr_std = np.std(arr, axis=0)
        for col in np.nonzero(np.abs(curr_std) < tol)[0]:
            bad_columns.add(col)
    good_columns = [i for i in range(arr_lst[0].shape[1]) if i not in bad_columns]
    return [arr[:, good_columns] for arr in arr_lst]


def robust_svd(arr, n_components, randomized=False, n_runs=1):
    """
    Do deterministic or randomized SVD on arr.

    Parameters
    ----------
    arr: np.array
        The array to do SVD on
    n_components: int
        Number of SVD components
    randomized: bool, default=False
        Whether to run randomized SVD
    n_runs: int, default=1
        Run multiple times and take the realization with the lowest Frobenious reconstruction error

    Returns
    -------
    u, s, vh: np.array
        u @ np.diag(s) @ vh is the reconstruction of the original arr
    """
    if randomized:
        best_err = float('inf')
        u, s, vh = None, None, None
        for _ in range(n_runs):
            curr_u, curr_s, curr_vh = randomized_svd(arr, n_components=n_components, random_state=None)
            curr_err = np.sum((arr - curr_u @ np.diag(curr_s) @ curr_vh) ** 2)
            if curr_err < best_err:
                best_err = curr_err
                u, s, vh = curr_u, curr_s, curr_vh
        assert u is not None and s is not None and vh is not None
    else:
        if n_runs > 1:
            warnings.warn("Doing deterministic SVD, n_runs reset to one.")
        u, s, vh = svds(arr*1.0, k=n_components) # svds can not handle integer values
    return u, s, vh

def svd_denoise(arr, n_components=20, randomized=False, n_runs=1):
    """
    Compute best rank-n_components approximation of arr by SVD.

    Parameters
    ----------
    arr: np.array of shape (n_samples, n_features)
        Data matrix
    n_components: int, default=20
        Number of components to keep
    randomized: bool, default=False
        Whether to use randomized SVD
    n_runs: int, default=1
        Run multiple times and take the realization with the lowest Frobenious reconstruction error

    Returns
    -------
    arr: array_like of shape (n_samples, n_features)
        Rank-n_comopnents approximation of the input arr.
    """
    if n_components is None:
        return arr
    u, s, vh = robust_svd(arr, n_components=n_components, randomized=randomized, n_runs=n_runs)
    return u @ np.diag(s) @ vh

def svd_embedding(arr, n_components=20, randomized=False, n_runs=1):
    """
    Compute rank-n_components SVD embeddings of arr.

    Parameters
    ----------
    arr: np.array of shape (n_samples, n_features)
        Data matrix
    n_components: int, default=20
        Number of components to keep
    randomized: bool, default=False
        Whether to use randomized SVD
    n_runs: int, default=1
        Run multiple times and take the realization with the lowest Frobenious reconstruction error

    Returns
    -------
    embeddings: array_like of shape (n_samples, n_components)
        Rank-n_comopnents SVD embedding of arr.
    """
    if n_components is None:
        return arr
    u, s, vh = robust_svd(arr, n_components=n_components, randomized=randomized, n_runs=n_runs)
    return u @ np.diag(s)



def center_scale(arr):
    """
    Column-wise center and scale by standard deviation.

    Parameters
    ----------
    arr: np.ndarray of shape (n_samples, n_features)

    Returns
    -------
    Center and scaled version of arr.
    """
    return (arr - arr.mean(axis=0)) / arr.std(axis=0)


def filter_bad_matches(matching, filter_prop=0.1):
    """
    Filter bad matches according to the distances of matched pairs.

    Parameters
    ----------
    matching: list
        rows, cols, vals = init_matching, where each matched pair is (rows[i], cols[i]),
        and their distance is vals[i]
    filter_prop: float
        Matched pairs with distance in top filter_prop are discarded
    Returns
    -------
    rows, cols, vals: list
        Each matched pair of rows[i], cols[i], their distance is vals[i]
    """
    init_rows, init_cols, init_vals = matching
    thresh = np.quantile(init_vals, 1 - filter_prop)
    rows = []
    cols = []
    vals = []
    for i, j, val in zip(init_rows, init_cols, init_vals):
        if val < thresh:
            rows.append(i)
            cols.append(j)
            vals.append(val)
    return np.array(rows, dtype=np.int32), np.array(cols, dtype=np.int32), np.array(vals, dtype=np.float32)

def pearson_correlation(arr1, arr2):
    """Calculate the vector of pearson correlations between each row of arr1 and arr2.

    Parameters
    ----------
    arr1: np.array of shape (n_samples, n_features)
        First dataset.
    arr2: np.array of shape (n_samples, n_features)
        Second dataset.

    Returns
    -------
    np.array of shape (n_samples,), the i-th entry is the pearson correlation between arr1[i, :] and arr2[i, :].
    """
    n, p = arr1.shape
    m, p2 = arr2.shape
    assert n == m and p2 == p

    arr1 = (arr1.T - np.mean(arr1, axis=1)).T
    arr2 = (arr2.T - np.mean(arr2, axis=1)).T

    arr1 = (arr1.T / np.sqrt(1e-6 + np.sum(arr1 ** 2, axis=1))).T
    arr2 = (arr2.T / np.sqrt(1e-6 + np.sum(arr2 ** 2, axis=1))).T

    return np.sum(arr1 * arr2, axis=1)

def cca_embedding(arr1, arr2, init_matching, filter_prop, n_components, max_iter=2000):
    """
    Filter bad matched pairs, align arr1 and arr2 using init_matching, fit CCA, and get CCA embeddings of arr1 and arr2.

    Parameters
    ----------
    arr1: np.ndarray of shape (n_samples1, n_features1)
        The first data matrix
    arr2: np.ndarray of shape (n_samples2, n_features2)
        The second data matrix
    init_matching: list
        rows, cols, vals = init_matching, where each matched pair is (rows[i], cols[i]),
        and their distance is vals[i]
    filter_prop: float
        Matched pairs with distance in top filter_prop are discarded when fitting CCA
    n_components: int
        Number of components to keep when fitting CCA
    max_iter: int, default=2000
        Maximum number of iterations for CCA

    Returns
    -------
    arr1_cca: np.array of shape (n_samples1, n_components)
    arr2_cca: np.array of shape (n_samples2, n_components)
    canonical_correlations: np.array of shape (n_components,)
    """

    # filter bad matched pairs
    arr1_indices, arr2_indices, _ = filter_bad_matches(init_matching, filter_prop)

    # align
    arr1_aligned = arr1[arr1_indices, :]
    arr2_aligned = arr2[arr2_indices, :]

    # cca
    cca = CCA(n_components=n_components, max_iter=max_iter)
    cca.fit(arr1_aligned, arr2_aligned)
    arr1_aligned_cca, arr2_aligned_cca = cca.transform(arr1_aligned, arr2_aligned)
    arr1_aligned_cca = center_scale(arr1_aligned_cca)
    arr2_aligned_cca = center_scale(arr2_aligned_cca)

    canonical_correlations = np.corrcoef(
        arr1_aligned_cca, arr2_aligned_cca, rowvar=False).diagonal(offset=n_components)
    arr1_cca, arr2_cca = cca.transform(arr1, arr2)
    arr1_cca = center_scale(arr1_cca)
    arr2_cca = center_scale(arr2_cca)

    return arr1_cca, arr2_cca, canonical_correlations


"""
Graph utility functions for protein
"""
import warnings
import numpy as np
from scipy.sparse.linalg import svds
from sklearn.utils.extmath import randomized_svd

def robust_svd(arr, n_components, randomized=False, n_runs=1):
    """
    Do deterministic or randomized SVD on arr.

    Parameters
    ----------
    arr: np.array
        The array to do SVD on
    n_components: int
        Number of SVD components
    randomized: bool, default=False
        Whether to run randomized SVD
    n_runs: int, default=1
        Run multiple times and take the realization with the lowest Frobenious reconstruction error

    Returns
    -------
    u, s, vh: np.array
        u @ np.diag(s) @ vh is the reconstruction of the original arr
    """
    if randomized:
        best_err = float('inf')
        u, s, vh = None, None, None
        for _ in range(n_runs):
            curr_u, curr_s, curr_vh = randomized_svd(arr, n_components=n_components, random_state=None)
            curr_err = np.sum((arr - curr_u @ np.diag(curr_s) @ curr_vh) ** 2)
            if curr_err < best_err:
                best_err = curr_err
                u, s, vh = curr_u, curr_s, curr_vh
        assert u is not None and s is not None and vh is not None
    else:
        if n_runs > 1:
            warnings.warn("Doing deterministic SVD, n_runs reset to one.")
        u, s, vh = svds(arr*1.0, k=n_components) # svds can not handle integer values
    return u, s, vh

def svd_embedding(arr, n_components=20, randomized=False, n_runs=1):
    """
    Compute rank-n_components SVD embeddings of arr.

    Parameters
    ----------
    arr: np.array of shape (n_samples, n_features)
        Data matrix
    n_components: int, default=20
        Number of components to keep
    randomized: bool, default=False
        Whether to use randomized SVD
    n_runs: int, default=1
        Run multiple times and take the realization with the lowest Frobenious reconstruction error

    Returns
    -------
    embeddings: array_like of shape (n_samples, n_components)
        Rank-n_comopnents SVD embedding of arr.
    """
    if n_components is None:
        return arr
    u, s, vh = robust_svd(arr, n_components=n_components, randomized=randomized, n_runs=n_runs)
    return u @ np.diag(s)

In [6]:
import numpy as np
from scipy.optimize import linear_sum_assignment

def match_cells(arr1, arr2, base_dist=None, wt_on_base_dist=0, verbose=True):
    """
    Get matching between arr1 and arr2 using linear assignment, the distance is 1 - Pearson correlation.

    Parameters
    ----------
    arr1: np.array of shape (n_samples1, n_features)
        The first data matrix
    arr2: np.array of shape (n_samples2, n_features)
        The second data matrix
    base_dist: None or np.ndarray of shape (n_samples1, n_samples2)
        Baseline distance matrix
    wt_on_base_dist: float between 0 and 1
        The final distance matrix to use is (1-wt_on_base_dist) * dist[arr1, arr2] + wt_on_base_dist * base_dist
    verbose: bool, default=True
        Whether to print the progress

    Returns
    -------
    rows, cols, vals: list
        Each matched pair of rows[i], cols[i], their distance is vals[i]
    """
    if verbose:
        print('Start the matching process...', flush=True)
        print('Computing the distance matrix...', flush=True)
    dist = cdist_correlation(arr1, arr2)
    if base_dist is not None:
        if verbose:
            print(
                f'Interpolating {1-wt_on_base_dist} * dist[arr1, arr2] + {wt_on_base_dist} * base_dist...',
                flush=True
            )
        dist = (1-wt_on_base_dist) * dist + wt_on_base_dist * base_dist
    if verbose:
        print('Solving linear assignment...', flush=True)
    rows, cols = linear_sum_assignment(dist)
    if verbose:
        print('Linear assignment completed!', flush=True)

    return rows, cols, np.array([dist[i, j] for i, j in zip(rows, cols)])


def get_initial_matching(
        arr1, arr2,
        randomized_svd=True,
        svd_runs=1,
        svd_components1=30, svd_components2=30,
        verbose=True
):
    """
    Assume the features of arr1 and arr2 are column-wise directly comparable,
    obtain a matching by minimizing the correlation distance between arr1 and arr2.

    Parameters
    ----------
    arr1: np.array of shape (n_samples1, n_features1)
        The first data matrix.
    arr2: np.array of shape (n_samples2, n_features2)
        The second data matrix.
    randomized_svd: bool, default=False
        Whether to use randomized svd.
    svd_runs: int, default=1
        Randomized SVD will result in different runs,
        so if randomized_svd=True, perform svd_runs many randomized SVDs, and pick the one with the
        smallest Frobenious reconstruction error.
        If randomized_svd=False, svd_runs is forced to be 1.
    svd_components1: None or int
        If None, then do not do SVD,
        else, number of components to keep when doing SVD de-noising for the first data matrix.
    svd_components2: None or int
        Same as svd_components1 but for the second data matrix.
    verbose: bool, default=True
        Whether to print the progress.

    Returns
    -------
    matching: list of length 3
        rows, cols, vals = matching,
        Each matched pair is rows[i], cols[i], their distance is vals[i].
    """
    # assert arr1.shape[1] == arr2.shape[1]
    arr1, arr2 = drop_zero_variability_columns(arr_lst=[arr1, arr2])

    if verbose:
        print('Normalizing and reducing the dimension of the data...', flush=True)
    arr1_svd = svd_embedding(
        arr=arr1, n_components=svd_components1,
        randomized=randomized_svd, n_runs=svd_runs
    )
    arr2_svd = svd_embedding(
        arr=arr2, n_components=svd_components2,
        randomized=randomized_svd, n_runs=svd_runs
    )

    res = match_cells(arr1=arr1_svd, arr2=arr2_svd, verbose=verbose)
    if verbose:
        print('Initial matching completed!', flush=True)

    return res

In [7]:
def find_initial_pivots(
        arr1, arr2,
        svd_components1=None, svd_components2=None,
        randomized_svd=False, svd_runs=1,
        verbose=True
):
    """
    Perform initial matching.

    Parameters
    ----------
    svd_components1: None or int, default=None
        If not None, perform SVD to reduce the dimension of self.shared_arr1.
    svd_components2: None or int, default=None
        If not None, perform SVD to reduce the dimension of self.shared_arr2.
    randomized_svd: bool, default=False
        Whether to use randomized SVD.
    svd_runs: int, default=1
        Perform multiple runs of SVD and the one with lowest Frobenious reconstruction error is selected.
    verbose: bool, default=True
        Whether to print the progress.

    Returns
    -------
    None
    """
    init_matching = []
    init_matching.append(
        get_initial_matching(
            arr1=arr1,
            arr2=arr2,
            randomized_svd=randomized_svd,
            svd_runs=svd_runs,
            svd_components1=svd_components1,
            svd_components2=svd_components2,
            verbose=False
        )
    )

    if verbose:
        print('Done!', flush=True)
    return init_matching


In [8]:
res = find_initial_pivots(
        rna.X, atac.X,
        svd_components1=30, svd_components2=30,
        randomized_svd=False, svd_runs=1,
        verbose=True
)

In [9]:

def get_refined_matching_one_iter(
        init_matching, arr1, arr2,
        filter_prop=0,
        cca_components=15,
        cca_max_iter=2000,
        verbose=True
):
    """
    Run one iteration of CCA refinement.

    Parameters
    ----------
    init_matching: list
        init_matching[0][i], init_matching[1][i] is a matched pair,
        and init_matching[2][i] is the distance for this pair
    arr1: np.array of shape (n_samples1, n_features1)
        The first data matrix.
    arr2: np.array of shape (n_samples2, n_features2)
        The second data matrix.
    filter_prop: float, default=0
        Proportion of matched pairs to discard before feeding into refinement iterations.
    cca_components: int, default=15
        Number of CCA components.
    cca_max_iter: int, default=2000
        Maximum number of CCA iterations.
    verbose: bool, default=True
        Whether to print the

    Returns
    -------
    rows, cols, vals: list
        Each matched pair of rows[i], cols[i], their distance is vals[i]
    """
    if verbose:
        print('Fitting CCA...', flush=True)
    arr1_cca, arr2_cca, _ = cca_embedding(
        arr1=arr1, arr2=arr2,
        init_matching=init_matching, filter_prop=filter_prop, n_components=cca_components, max_iter=cca_max_iter
    )

    return match_cells(
        arr1=arr1_cca, arr2=arr2_cca, verbose=verbose
    )

def get_refined_matching(
        init_matching, arr1, arr2,
        randomized_svd=False, svd_runs=1,
        svd_components1=None, svd_components2=None,
        n_iters=3, filter_prop=0,
        cca_components=20,
        cca_max_iter=2000,
        verbose=True
):
    ns = [len(x) for x in init_matching]
    assert ns[0] == ns[1] == ns[2]
    assert isinstance(n_iters, int) and n_iters >= 1
    assert 0 <= int(ns[0] * filter_prop) < ns[0]

    assert 1 <= cca_components <= min(arr1.shape[1], arr2.shape[1])

    # arr1 = drop_zero_variability_columns(arr_lst=[arr1])[0]
    # arr2 = drop_zero_variability_columns(arr_lst=[arr2])[0]

    # denoising
    if verbose:
        print("Denoising the data...", flush=True)
    arr1 = svd_denoise(
        arr=arr1, n_components=svd_components1, randomized=randomized_svd,
        n_runs=svd_runs
    )
    arr2 = svd_denoise(
        arr=arr2, n_components=svd_components2, randomized=randomized_svd,
        n_runs=svd_runs
    )

    # prepare the distance matrix used in the initial matching
    cca_matching = init_matching
    # iterative refinement
    for it in range(n_iters):
        if verbose:
            print('Now at iteration {}:'.format(it), flush=True)
        cca_matching = get_refined_matching_one_iter(
            init_matching=cca_matching,
            arr1=arr1, arr2=arr2,
            filter_prop=filter_prop,
            cca_components=cca_components,
            cca_max_iter=cca_max_iter, verbose=verbose
        )

    if verbose:
        print('Refined matching completed!', flush=True)
    return cca_matching

In [10]:

def refine_pivots(
        init_matching, arr1, arr2,
        svd_components1=None, svd_components2=None,
        cca_components=None,
        filter_prop=0,
        n_iters=1,
        randomized_svd=False, svd_runs=1,
        cca_max_iter=2000,
        verbose=True
):
    refined_matching = []
    refined_matching.append(
        get_refined_matching(
            init_matching=init_matching,
            arr1=arr1,
            arr2=arr2,
            randomized_svd=randomized_svd,
            svd_runs=svd_runs,
            svd_components1=svd_components1,
            svd_components2=svd_components2,
            n_iters=n_iters,
            filter_prop=filter_prop,
            cca_components=cca_components,
            cca_max_iter=cca_max_iter,
            verbose=False
        )
    )

    if verbose:
        print('Done!', flush=True)
    return refined_matching


In [11]:
import sklearn
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.decomposition import TruncatedSVD
import scipy
from scipy.sparse import csr_matrix
from scipy.sparse import issparse, csr

def tfidf(X, n_components, binarize=True, random_state=0):

    sc_count = np.copy(X)
    if binarize:
        sc_count = np.where(sc_count < 1, sc_count, 1)

    tfidf = TfidfTransformer(norm='l2', sublinear_tf=True)
    normed_count = tfidf.fit_transform(sc_count)
    lsi = TruncatedSVD(n_components=n_components, random_state=random_state)
    lsi_r = lsi.fit_transform(normed_count)

    X_lsi = lsi_r[:, 1:]

    return normed_count, X_lsi

def TFIDF_LSI(adata, n_comps=50, binarize=True, random_state=0):
    '''
    Computes LSI based on a TF-IDF transformation of the data from MultiMap. Putative dimensionality
    reduction for scATAC-seq data. Adds an ``.obsm['X_lsi']`` field to the object it was ran on.

    Input
    -----
    adata : ``AnnData``
        The object to run TFIDF + LSI on. Will use ``.X`` as the input data.
    n_comps : ``int``
        The number of components to generate. Default: 50
    binarize : ``bool``
        Whether to binarize the data prior to the computation. Often done during scATAC-seq
        processing. Default: True
    random_state : ``int``
        The seed to use for randon number generation. Default: 0
    '''

    # this is just a very basic wrapper for the non-adata function
    import scipy
    if scipy.sparse.issparse(adata.X):
        adata.X, adata.obsm['X_lsi'] = tfidf(adata.X.todense(), n_components=n_comps, binarize=binarize,
                                    random_state=random_state)
    else:
        adata.X, adata.obsm['X_lsi'] = tfidf(adata.X, n_components=n_comps, binarize=binarize, random_state=random_state)
        
def preprocessing_atac(adata,
                       data_type: str = None,
                       genome: str = None,
                       use_gene_weigt: bool = True,
                       use_top_pcs: bool = False,
                       min_features: int = 100,
                       min_cells: int = 3,
                       atac_n_top_features=100000,  # or gene list
                       n: int = 15,
                       batch_key: str = 'batch',
                       metric: str = 'euclidean',
                       n_components: int = 50
                       ):
    """
    Preprocess scCAS data matrix.

    Parameters
    ----------
    adata :  AnnData object of shape `n_obs` × `n_vars`. Rows correspond to cells and columns to peaks/regions.
    filter_rate : float, optional
        Proportion for feature selection, by default 0.01
    """
    if min_features is None: min_features = 100
    if atac_n_top_features is None: atac_n_top_features = 100000

    # Preprocessing
    # if type(adata.X) != csr.csr_matrix:
    #     adata.X = scipy.sparse.csr_matrix(adata.X)

    sc.pp.filter_cells(adata, min_genes=min_features)
    sc.pp.filter_genes(adata, min_cells=min_cells)
    # epi.pp.filter_cells(adata, min_features=min_features)
    # epi.pp.filter_features(adata, min_cells=min_cells)
    adata.raw = adata

    ## TFIDF & LSI
    TFIDF_LSI(adata, n_comps=n_components, binarize=False, random_state=0)

    # use scanpy functions to do the graph construction
    # sc.pp.neighbors(adata, n_neighbors=n, metric=metric, use_rep='X_lsi')

    ## HVP
    if type(atac_n_top_features) == int and atac_n_top_features > 0 and atac_n_top_features < adata.shape[1]:
        sc.pp.highly_variable_genes(adata, n_top_genes=atac_n_top_features, batch_key=batch_key,
                                    inplace=False, subset=True)

    return adata

In [12]:
atac

In [13]:
atac = preprocessing_atac(atac,
                       min_features= 100,
                       min_cells = 3,
                       atac_n_top_features=10000,  # or gene list
                       )
atac

In [14]:
res2 = refine_pivots(
        res[0], rna.X, atac.X,
        svd_components1=30, svd_components2=30,
        cca_components=20,
        filter_prop=0,
        n_iters=1,
        randomized_svd=False, svd_runs=1,
        cca_max_iter=2000,
        verbose=True
)

In [15]:
res[0]

In [16]:
res2[0]

In [17]:

def fit_svd_on_full_data(arr1, arr2, randomized_svd=False, svd_runs=1,
                         svd_components1=None, svd_components2=None):
    """Perform SVD on full self.active_arr1 and self.active_arr2 and save the functions that reduce the dimension
    of the data.
    """
    if svd_components1 is not None:
        u1, s1, vh1 = robust_svd(
            arr=arr1, n_components=svd_components1,
            randomized=randomized_svd, n_runs=svd_runs
        )
        rotation_before_cca_on_active_arr1 = lambda arr: arr @ vh1.T
    else:
        rotation_before_cca_on_active_arr1 = lambda arr: arr

    if svd_components2 is not None:
        u2, s2, vh2 = robust_svd(
            arr=arr2, n_components=svd_components2,
            randomized=randomized_svd, n_runs=svd_runs
        )
        rotation_before_cca_on_active_arr2 = lambda arr: arr @ vh2.T
    else:
        rotation_before_cca_on_active_arr2 = lambda arr: arr

    return rotation_before_cca_on_active_arr1, rotation_before_cca_on_active_arr2

In [91]:
from collections import defaultdict
from sklearn.cross_decomposition import CCA

def filter_bad_matches(arr1, arr2, refined_matching=None, propagated_matching=None, target='pivot', filter_prop=0.,
                       randomized_svd=False, svd_runs=1,
                       svd_components1=None, svd_components2=None,
                       cca_components=20,
                       cca_max_iter=2000,
                       verbose=True):
    # construct mapping from metacell index to single cell indices
    # useful when stitching the batch-wise matchings together
    metacell_idx_to_array_indices = []

    arr1_raw = arr1
    arr2_raw = arr2
    # conduct filtering
    n_remaining = 0
    if verbose:
        print('Begin filtering...', flush=True)
    if target == 'pivot':
        matching_to_be_filtered = refined_matching
    elif target == 'propagated':
        matching_to_be_filtered = propagated_matching
    else:
        raise ValueError('target must be in {\'pivot\', \'propagated\'}.')

    remaining_indices_after_filtering = []
    idx2_to_indices1 = defaultdict(set)
    # record the locations that survive the filtering
    rows, cols, vals = matching_to_be_filtered[0]
    # anything with val <= thresh will be retained
    thresh = np.quantile(vals, 1 - filter_prop)
    remaining_indices_after_filtering.append([i for i in range(len(vals)) if vals[i] <= thresh])
    n_remaining += len(remaining_indices_after_filtering[0])
    for i in remaining_indices_after_filtering[0]:
        idx1, idx2 = rows[i], cols[i]
        idx2_to_indices1[idx2].add(idx1)

    if target == 'pivot':
        remaining_indices_in_refined_matching = remaining_indices_after_filtering[0]
    else:
        remaining_indices_in_propagated_matching = remaining_indices_after_filtering[0]

    if verbose:
        print('{}/{} pairs of matched cells remain after the filtering.'.format(
            n_remaining, np.sum([len(per_batch_matching[0]) for per_batch_matching in matching_to_be_filtered])
        ))

    # convert idx2_to_indices1 to a dict of lists for ease of later usage
    idx2_to_indices1 = {idx2: sorted(indices1) for idx2, indices1 in idx2_to_indices1.items()}

    # fit CCA on pivots
    if verbose:
        print('Fitting CCA on pivots...', flush=True)
    rotation_before_cca_on_active_arr1, rotation_before_cca_on_active_arr2 = fit_svd_on_full_data(arr1, arr2, randomized_svd=randomized_svd, svd_runs=svd_runs,
                                                                                                  svd_components1=svd_components1, svd_components2=svd_components2)
    arr1_list, arr2_list = [], []
    for idx2, indices1 in idx2_to_indices1.items():
        arr1_list.append(arr1[indices1[0], :]) # np.mean(self.active_arr1[indices1, :], axis=0)
        arr2_list.append(arr2[idx2, :].toarray().squeeze())
    arr1 = rotation_before_cca_on_active_arr1(np.array(arr1_list))
    arr2 = rotation_before_cca_on_active_arr2(np.array(arr2_list))
    
    arr1_raw = rotation_before_cca_on_active_arr1(arr1_raw)
    arr2_raw = rotation_before_cca_on_active_arr2(arr2_raw)
    
    cca_on_pivots = CCA(n_components=cca_components, max_iter=cca_max_iter)
    cca_on_pivots.fit(arr1, arr2)

    if verbose:
        print('Scoring matched pairs...', flush=True)
    # transform the whole dataset
    arr1, arr2 = cca_on_pivots.transform(arr1_raw, arr2_raw)
    arr1 = center_scale(arr1)
    arr2 = center_scale(arr2)

    # add distances to idx2_to_indices1
    all_indices1, all_indices2 = [], []
    for idx2, indices1 in idx2_to_indices1.items():
        for idx1 in indices1:
            all_indices1.append(idx1)
            all_indices2.append(idx2)
    # compute all possible pairs of distances
    pearson_correlations = pearson_correlation(arr1[all_indices1, :], arr2[all_indices2, :])

    cnt = 0
    for idx2, indices1 in idx2_to_indices1.items():
        indices_and_scores = []
        for idx1 in indices1:
            indices_and_scores.append((idx1, pearson_correlations[cnt]))
            cnt += 1
        idx2_to_indices1[idx2] = indices_and_scores

    if target == 'pivot':
        pivot2_to_pivots1 = idx2_to_indices1
        return pivot2_to_pivots1, remaining_indices_in_refined_matching
    else:
        propidx2_to_propindices1 = idx2_to_indices1
        return propidx2_to_propindices1, remaining_indices_in_propagated_matching

In [92]:
pivot2_to_pivots1, remaining_indices_in_refined_matching = filter_bad_matches(rna.X, atac.X, refined_matching=res2, target='pivot', filter_prop=0.3,
                       randomized_svd=False, svd_runs=1,
                       svd_components1=30, svd_components2=30,
                       cca_components=20,
                       cca_max_iter=2000,
                       verbose=True)

In [95]:
len(remaining_indices_in_refined_matching)

In [149]:
import pynndescent

import numpy as np
from scipy.sparse import csr_matrix

def convert_to_numpy(arr):
    if isinstance(arr, csr_matrix):
        return arr.toarray()
    elif isinstance(arr, np.ndarray):
        return arr
    else:
        raise TypeError("Unsupported data type.")

def get_nearest_neighbors(query_arr, target_arr, svd_components=None, randomized_svd=False, svd_runs=1,
                          metric='correlation'):
    """
    For each row in query_arr, compute its nearest neighbor in target_arr.

    Parameters
    ----------
    query_arr: np.array of shape (n_samples1, n_features)
        The query data matrix.
    target_arr: np.array of shape (n_samples2, n_features)
        The target data matrix.
    svd_components: None or int, default=None
        If not None, will first conduct SVD to reduce the dimension
        of the vertically stacked version of query_arr and target_arr.
    randomized_svd: bool, default=False
        Whether to use randomized SVD.
    svd_runs: int, default=1
        Run multiple instances of SVD and select the one with the lowest Frobenious reconstruction error.
    metric: string, default='correlation'
        The metric to use in nearest neighbor search.

    Returns
    -------
    neighbors: np.array of shape (n_samples1)
        The i-th element is the index in target_arr to whom the i-th row of query_arr is closest to.
    dists: np.array of shape (n_samples1)
        The i-th element is the distance corresponding to neighbors[i].
    """
    query_arr = convert_to_numpy(query_arr)
    target_arr = convert_to_numpy(target_arr)
    arr = np.vstack([query_arr, target_arr])
    arr = svd_embedding(
        arr=arr, n_components=svd_components,
        randomized=randomized_svd,
        n_runs=svd_runs
    )
    query_arr = arr[:query_arr.shape[0], :]
    pivot_arr = arr[query_arr.shape[0]:, :]
    # approximate nearest neighbor search
    index = pynndescent.NNDescent(pivot_arr, n_neighbors=100, metric=metric)
    neighbors, dists = index.query(query_arr, k=50)
    neighbors, dists = neighbors[:, 0], dists[:, 0]
    return neighbors, dists


def propagate(
        refined_matching=None, remaining_indices_in_refined_matching=None, arr1=None, arr2=None, svd_components1=None, svd_components2=None,
        metric='euclidean',
        randomized_svd=False, svd_runs=1, verbose=True
):
    """
    For indices not in pivots, find their matches by nearest neighbor search.

    Parameters
    ----------
    svd_components1: None or int, default=None
        If not None, perform SVD to reduce the dimension of self.active_arr1
        before doing internal nearest neighbor search.
    svd_components2: None or int, default=None
        If not None, perform SVD to reduce the dimension of self.active_arr1
        before doing internal nearest neighbor search.
    wt1: float, default=0.7
        Weight to put on raw data of self.active_arr1 when doing smoothing.
    wt2: float, default=0.7
        Weight to put on raw data of self.active_arr2 when doing smoothing.
    metric: string, default='correlation'
        The metric to use in nearest neighbor search.
    randomized_svd: bool, default=False
        Whether to perform randomized SVD.
    svd_runs: int, default=1
        Perform multiple runs of SVD and the one with lowest Frobenious reconstruction error is selected.
    verbose: bool, default=True
        Whether to print the progress.

    Returns
    -------
    None
    """
    propagated_matching = []
    curr_propagated_matching = [[], [], []]
    curr_refined_matching = refined_matching
    # get good pivot indices that survived pivot filtering
    existing_indices = curr_refined_matching[0][remaining_indices_in_refined_matching]
    good_indices1 = curr_refined_matching[0][existing_indices]
    good_indices2 = curr_refined_matching[1][existing_indices]

    # get arrays that were used when doing refined matching
    # get remaining indices
    # propagation will only be done for those indices
    good_indices1_set = set(good_indices1)
    remaining_indices1 = [i for i in range(arr1.shape[0]) if i not in good_indices1_set]
    good_indices2_set = set(good_indices2)
    remaining_indices2 = [i for i in range(arr2.shape[0]) if i not in good_indices2_set]

    # propagate for remaining indices in arr1
    if len(remaining_indices1) > 0:
        # get 1-nearest-neighbors and the corresponding distances
        remaining_indices1_nns, remaining_indices1_nn_dists = get_nearest_neighbors(
            query_arr=arr1[remaining_indices1, :],
            target_arr=arr1[good_indices1, :],
            svd_components=svd_components1,
            randomized_svd=randomized_svd,
            svd_runs=svd_runs,
            metric=metric
        )
        matched_indices2 = good_indices2[remaining_indices1_nns]
        curr_propagated_matching[0].extend(remaining_indices1)
        curr_propagated_matching[1].extend(matched_indices2)
        curr_propagated_matching[2].extend(remaining_indices1_nn_dists)

    # propagate for remaining indices in arr2
    if len(remaining_indices2) > 0:
        # get 1-nearest-neighbors and the corresponding distances
        remaining_indices2_nns, remaining_indices2_nn_dists = get_nearest_neighbors(
            query_arr=arr2[remaining_indices2, :],
            target_arr=arr2[good_indices2, :],
            svd_components=svd_components2,
            randomized_svd=randomized_svd,
            svd_runs=svd_runs,
            metric=metric
        )
        matched_indices1 = good_indices1[remaining_indices2_nns]
        curr_propagated_matching[0].extend(matched_indices1)
        curr_propagated_matching[1].extend(remaining_indices2)
        curr_propagated_matching[2].extend(remaining_indices2_nn_dists)

    propagated_matching.append(
        (np.array(curr_propagated_matching[0]),
         np.array(curr_propagated_matching[1]),
         np.array(curr_propagated_matching[2]))
    )

    if verbose:
        print('Done!', flush=True)

    return propagated_matching

In [150]:
res2[0]

In [151]:
propagated_matching = propagate(
    refined_matching=res2[0], remaining_indices_in_refined_matching=remaining_indices_in_refined_matching, 
    arr1=rna.X, arr2=atac.X, svd_components1=30, svd_components2=30, metric='euclidean',
    randomized_svd=False, svd_runs=1, verbose=True
)

In [152]:
propagated_matching

In [155]:
len(propagated_matching[0][2])

In [158]:
def address_matching_redundancy(matching, order=(1, 2)):
    """
    Make a potentially multiple-to-multiple matching to an one-to-one matching according to order.

    Parameters
    ----------
    matching: list of length three
        rows, cols, vals = matching: list
        Each matched pair of rows[i], cols[i], their score (the larger, the better) is vals[i]
    order: None or (1, 2) or (2, 1)
        If None, do nothing;
        If (1, 2), then the redundancy is addressed by making matching
        an injective map from the first dataset to the second;
        if (2, 1), do the other way around.

    Returns
    -------
    rows, cols, vals: list
        Each matched pair of rows[i], cols[i], their score is vals[i].
    """
    if order is None:
        return matching
    res = [[], [], []]
    if order == (1, 2):
        idx1_to_idx2 = dict()
        idx1_to_score = dict()
        for i, j, score in zip(matching[0], matching[1], matching[2]):
            if i not in idx1_to_idx2:
                idx1_to_idx2[i] = j
                idx1_to_score[i] = score
            elif score > idx1_to_score[i]:
                idx1_to_idx2[i] = j
                idx1_to_score[i] = score
        for idx1, idx2 in idx1_to_idx2.items():
            res[0].append(idx1)
            res[1].append(idx2)
            res[2].append(idx1_to_score[idx1])
    elif order == (2, 1):
        idx2_to_idx1 = dict()
        idx2_to_score = dict()
        for i, j, score in zip(matching[0], matching[1], matching[2]):
            if j not in idx2_to_idx1:
                idx2_to_idx1[j] = i
                idx2_to_score[j] = score
            elif score > idx2_to_score[j]:
                idx2_to_idx1[j] = i
                idx2_to_score[j] = score
        for idx2, idx1 in idx2_to_idx1.items():
            res[0].append(idx1)
            res[1].append(idx2)
            res[2].append(idx2_to_score[idx2])
    else:
        raise NotImplementedError('order must be in {None, (1, 2), (2, 1)}.')

    return res


def get_matching(pivot2_to_pivots1=None, propidx2_to_propindices1=None, arr1=None, arr2=None, order=None, target='pivot'):
    """
    Return a copy of the desired matching.

    Parameters
    ----------
    order: None or (1, 2) or (1, 2), default=None
        If (1, 2), then every cell in target arr1 has at least one match
        if (2, 1), then does the other way around,
        if None, then every cell in target arr1 and every cell in target arr2 both have at least one match
    target: 'pivot' or 'full_data'
        If 'pivot', then only return matching on pivots, else return matching on all the data.

    Returns
    -------
    A matching of format dict or list.
    """
    if target not in {'pivot', 'full_data'}:
        raise ValueError('mode must be in {\'pivot_only\', \'full_data\'}.')
    # if return_format not in {'dict', 'list'}:
    #     raise ValueError('return_format must be in {\'dict\', \'list\'}.')
    res = [[], [], []]
    for idx2, indices1_and_scores in pivot2_to_pivots1.items():
        for idx1, score in indices1_and_scores:
            res[0].append(idx1)
            res[1].append(idx2)
            res[2].append(score)
    if target == 'pivot':
        return res
    elif target == 'full_data':
        if order == (1, 2):
            # add propagated matching for non-pivot cells in the first dataset
            existing_indices1 = np.unique(res[0])
            remaining_indices1 = [i for i in range(arr1.shape[0]) if i not in existing_indices1]
            propagated_idx1_to_indices2 = defaultdict(list)
            for idx2, indices1_and_scores in propidx2_to_propindices1.items():
                for idx1, score in indices1_and_scores:
                    propagated_idx1_to_indices2[idx1].append((idx2, score))
            for idx1 in remaining_indices1:
                if idx1 in propagated_idx1_to_indices2:
                    for idx2, score in propagated_idx1_to_indices2[idx1]:
                        res[0].append(idx1)
                        res[1].append(idx2)
                        res[2].append(score)
        elif order == (2, 1):
            # add propagated matching for non-pivot cells in the second dataset
            existing_indices2 = np.unique(res[1])
            remaining_indices2 = [i for i in range(arr2.shape[0]) if i not in existing_indices2]
            for idx2 in remaining_indices2:
                if idx2 in propidx2_to_propindices1:
                    for idx1, score in propidx2_to_propindices1[idx2]:
                        res[0].append(idx1)
                        res[1].append(idx2)
                        res[2].append(score)
        elif order is None:
            # first do order (1, 2) and then do order (2, 1)
            # add propagated matching for non-pivot cells in the first dataset
            existing_indices1 = np.unique(res[0])
            remaining_indices1 = [i for i in range(arr1.shape[0]) if i not in existing_indices1]
            propagated_idx1_to_indices2 = defaultdict(list)
            for idx2, indices1_and_scores in propidx2_to_propindices1.items():
                for idx1, score in indices1_and_scores:
                    propagated_idx1_to_indices2[idx1].append((idx2, score))
            for idx1 in remaining_indices1:
                if idx1 in propagated_idx1_to_indices2:
                    for idx2, score in propagated_idx1_to_indices2[idx1]:
                        res[0].append(idx1)
                        res[1].append(idx2)
                        res[2].append(score)

            # add propagated matching for non-pivot cells in the second dataset
            existing_indices2 = np.unique(res[1])
            remaining_indices2 = [i for i in range(arr2.shape[0]) if i not in existing_indices2]
            for idx2 in remaining_indices2:
                if idx2 in propidx2_to_propindices1:
                    for idx1, score in propidx2_to_propindices1[idx2]:
                        res[0].append(idx1)
                        res[1].append(idx2)
                        res[2].append(score)

        else:
            raise NotImplementedError('order must be None or (1, 2) or (2, 1).')
    else:
        raise NotImplementedError('target must be in {\'pivot\', \'full_data\'}.')

    return address_matching_redundancy(matching=res, order=order)

In [166]:
propidx2_to_propindices1, remaining_indices_in_propagated_matching = filter_bad_matches(rna.X, atac.X, propagated_matching=propagated_matching, target='propagated', filter_prop=0,
                       randomized_svd=False, svd_runs=1,
                       svd_components1=30, svd_components2=30,
                       cca_components=20,
                       cca_max_iter=2000,
                       verbose=True)

In [168]:
len(propidx2_to_propindices1)

In [181]:
final_cellmatching = get_matching(pivot2_to_pivots1=pivot2_to_pivots1, propidx2_to_propindices1=propidx2_to_propindices1, 
                                  arr1=rna.X, arr2=atac.X, order=None, target='full_data')
# final_cellmatching

In [182]:
len(final_cellmatching[0])

In [189]:
df = pd.DataFrame(list(zip(final_cellmatching[0], final_cellmatching[1], final_cellmatching[2])),
             columns = ['mod1_indx', 'mod2_indx', 'score'])#['score'].min()
# columns: cell idx in mod1, cell idx in mod2, and matching scores

In [190]:
df

In [191]:
df['mod2_indx'] = df['mod2_indx'] + len(df['mod1_indx'])
df